In [1]:
import json
from pathlib import Path

# 프로젝트 루트
BASE_DIR = Path("/Users/wnsgud/workplace/multiturn-rag")

# 파일 경로들
question_path = BASE_DIR / "data" / "processed" / "question.jsonl"
rag_path = BASE_DIR / "data" / "raw" / "RAG.jsonl"
output_path = BASE_DIR / "data" / "processed" / "question_with_targets.jsonl"

# 1) RAG.jsonl에서 task_id -> targets.text 매핑 만들기
rag_target_map = {}

with rag_path.open("r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            print(f"[RAG.jsonl] JSON decode error at line {line_num}")
            continue

        task_id = obj.get("task_id")
        targets = obj.get("targets", [])

        # targets 안의 text만 추출
        target_texts = []
        if isinstance(targets, list):
            for t in targets:
                if isinstance(t, dict):
                    text = t.get("text")
                    if text is not None:
                        target_texts.append(text)

        # 보통 1개일 가능성이 크지만, 여러 개면 리스트 그대로 보존
        if task_id is not None:
            if len(target_texts) == 0:
                rag_target_map[task_id] = None
            elif len(target_texts) == 1:
                rag_target_map[task_id] = target_texts[0]
            else:
                rag_target_map[task_id] = target_texts  # 여러 개면 리스트로 저장

print(f"RAG 매핑 개수: {len(rag_target_map)}")

# 2) question.jsonl 읽어서 query-id 기준으로 target text 붙이기
matched = 0
total = 0

with question_path.open("r", encoding="utf-8") as fin, \
     output_path.open("w", encoding="utf-8") as fout:

    for line_num, line in enumerate(fin, start=1):
        line = line.strip()
        if not line:
            continue

        total += 1

        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            print(f"[question.jsonl] JSON decode error at line {line_num}")
            continue

        query_id = obj.get("query-id")

        # 새 컬럼 추가
        obj["target_text"] = rag_target_map.get(query_id)

        if query_id in rag_target_map:
            matched += 1

        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"총 question 개수: {total}")
print(f"매칭된 개수: {matched}")
print(f"저장 완료: {output_path}")

RAG 매핑 개수: 842
총 question 개수: 180
매칭된 개수: 180
저장 완료: /Users/wnsgud/workplace/multiturn-rag/data/processed/question_with_targets.jsonl
